In [46]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def rsibb(df):
    """ 
    Processes the given DataFrame:
    - Converts timestamp to datetime
    - Converts numeric columns to float
    - Calculates Bollinger Bands & RSI
    - Returns the processed DataFrame and the figure object
    """

    # Convert timestamp to datetime
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")

    # Convert numeric columns to float
    numeric_cols = ["open", "high", "low", "close", "volume"]
    df[numeric_cols] = df[numeric_cols].astype(float)

    # --- Calculate Bollinger Bands ---
    window, std_dev = 20, 2
    df['SMA'] = df['close'].rolling(window=window).mean()
    df['Upper_BB'] = df['SMA'] + (df['close'].rolling(window=window).std() * std_dev)
    df['Lower_BB'] = df['SMA'] - (df['close'].rolling(window=window).std() * std_dev)

    # --- Calculate RSI ---
    period = 14
    delta = df['close'].diff(1)  # Daily change
    gain = np.where(delta > 0, delta, 0)  # Positive gains
    loss = np.where(delta < 0, -delta, 0)  # Negative losses

    avg_gain = pd.Series(gain).rolling(window=period, min_periods=1).mean()
    avg_loss = pd.Series(loss).rolling(window=period, min_periods=1).mean()

    rs = avg_gain / avg_loss  # Relative Strength
    df['RSI'] = 100 - (100 / (1 + rs))  # RSI Formula

    # --- Create Subplots (Candlestick + RSI) ---
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        row_heights=[0.7, 0.3], vertical_spacing=0.1, 
)

    # --- Candlestick Chart ---
    fig.add_trace(go.Candlestick(
        x=df["open_time"],
        open=df["open"],
        high=df["high"],
        low=df["low"],
        close=df["close"],
        name="BTC/USDT",
        increasing_line=dict(width=0.5, color="green"),  # Up candles
        decreasing_line=dict(width=0.5, color="red")     # Down candles
    ), row=1, col=1)

    # --- Bollinger Bands ---
    fig.add_trace(go.Scatter(
        x=df["open_time"], y=df["Upper_BB"], name="Upper BB", line=dict(color="green", width=1), opacity=0.5
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=df["open_time"], y=df["Lower_BB"], name="Lower BB", line=dict(color="red", width=1), opacity=0.5
    ), row=1, col=1)

    # --- RSI Line Plot ---
    fig.add_trace(go.Scatter(
        x=df["open_time"], y=df["RSI"], name="RSI", line=dict(color="yellow", width=1.5)
    ), row=2, col=1)

    # --- Add RSI Overbought (70) & Oversold (30) Lines ---
    fig.add_hline(y=70, line_dash="dot", line_color="red", row=2, col=1)
    fig.add_hline(y=30, line_dash="dot", line_color="green", row=2, col=1)

    # --- Customize Layout ---
    fig.update_layout(
        xaxis_title="Time",
        yaxis_title="Price (BTC)",
        xaxis_rangeslider_visible=False,
        template="plotly_dark",
        height=800
    )

    return df, fig  # Return processed DataFrame and figure object





In [47]:
# --- Load Data ---
df = pd.read_csv('../database/btcusdt/1d.csv')

# Process data using rsibb function
df, fig = rsibb(df)

# Show the interactive chart
fig.show()

In [44]:
eurusd = pd.read_excel("../database/eurusd/eurusd2022.xlsx")
eurusd.columns = ["open_time", "open", "high", "low", "close", "volume"]
eurusd = eurusd.drop(columns=["volume"])

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:226: UserWarning:

Workbook contains no default style, apply openpyxl's default



In [48]:
eurusd

,open_time,open,high,low,close
0,2022-01-02 17:04:00,1.13689,1.13689,1.13689,1.13689
1,2022-01-02 17:05:00,1.13692,1.13692,1.13692,1.13692
2,2022-01-02 17:06:00,1.13668,1.13671,1.13652,1.13652
3,2022-01-02 17:07:00,1.13655,1.13692,1.13649,1.13682
4,2022-01-02 17:08:00,1.13682,1.13682,1.13682,1.13682
...,...,...,...,...,...
372859,2022-12-30 16:54:00,1.06997,1.07021,1.06981,1.07001
372860,2022-12-30 16:55:00,1.07001,1.07017,1.06989,1.07001
372861,2022-12-30 16:56:00,1.07004,1.07012,1.07003,1.07008
372862,2022-12-30 16:57:00,1.07003,1.07030,1.06995,1.07030


In [50]:
# df = pd.read_excel("../database/eurusd/eurusd2022.xlsx")
# df.columns = ["open_time", "open", "high", "low", "close", "volume"]
# df = df.drop(columns=["volume"])

# # Convert timestamp to datetime
# df["open_time"] = pd.to_datetime(df["open_time"])

# # Convert numeric columns to float
# numeric_cols = ["open", "high", "low", "close"]
# df[numeric_cols] = df[numeric_cols].astype(float)

# # --- Calculate Bollinger Bands ---
# window, std_dev = 20, 2
# df['SMA'] = df['close'].rolling(window=window).mean()
# df['Upper_BB'] = df['SMA'] + (df['close'].rolling(window=window).std() * std_dev)
# df['Lower_BB'] = df['SMA'] - (df['close'].rolling(window=window).std() * std_dev)

# # --- Calculate RSI ---
# period = 14
# delta = df['close'].diff(1)
# gain = np.where(delta > 0, delta, 0)
# loss = np.where(delta < 0, -delta, 0)

# avg_gain = pd.Series(gain).rolling(window=period, min_periods=1).mean()
# avg_loss = pd.Series(loss).rolling(window=period, min_periods=1).mean()

# rs = avg_gain / avg_loss
# df['RSI'] = 100 - (100 / (1 + rs))

# # --- Create Subplots (Candlestick + RSI) ---
# fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.7, 0.3], vertical_spacing=0.1)

# # --- Candlestick Chart ---
# fig.add_trace(go.Candlestick(
#     x=df["open_time"],
#     open=df["open"],
#     high=df["high"],
#     low=df["low"],
#     close=df["close"],
#     name="EUR/USD",
#     increasing_line=dict(width=0.5, color="green"),
#     decreasing_line=dict(width=0.5, color="red")
# ), row=1, col=1)

# # --- Bollinger Bands ---
# fig.add_trace(go.Scatter(x=df["open_time"], y=df["Upper_BB"], name="Upper BB", line=dict(color="green", width=1), opacity=0.5), row=1, col=1)
# fig.add_trace(go.Scatter(x=df["open_time"], y=df["Lower_BB"], name="Lower BB", line=dict(color="red", width=1), opacity=0.5), row=1, col=1)

# # --- RSI Line Plot ---
# fig.add_trace(go.Scatter(x=df["open_time"], y=df["RSI"], name="RSI", line=dict(color="yellow", width=1.5)), row=2, col=1)

# # --- Add RSI Overbought (70) & Oversold (30) Lines ---
# fig.add_hline(y=70, line_dash="dot", line_color="red", row=2, col=1)
# fig.add_hline(y=30, line_dash="dot", line_color="green", row=2, col=1)

# # --- Customize Layout ---
# fig.update_layout(
#     xaxis_title="Time",
#     yaxis_title="Price (EUR/USD)",
#     xaxis_rangeslider_visible=False,
#     template="plotly_dark",
#     height=800
# )

# # Show the interactive chart
# fig.show()